# SpecMine results for a paper

This notebook retrains all five baselines on one reproducible, repository-disjoint
holdout. Scores measure **agreement with supplied labels**, not verified
authorship. The default labels and their confidence scores are AI-assigned.
Model scores and labeler confidence are different quantities; neither proves
whether an LLM was used. Small samples and correlated features can make
estimates and rankings unstable.

Run top to bottom. Figures are displayed and saved as 300-dpi PNG and vector
PDF in a unique run directory under `notebooks/figures/` after all cells succeed.
Missing cached data may be downloaded unless
`LOCAL_DATA_ONLY` is enabled. Change `SEED`, `TEXT_FEATURE`, or the paths below
to configure the analysis.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

project_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "src" / "specmine").is_dir()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        "Could not find src/specmine. Open this notebook from the project "
        "folder or one of its subfolders."
    )
sys.path.insert(0, str(project_root / "src"))

from specmine.data import load_data, load_feature_data
from specmine.diagnostics import InsufficientFoldSupport, logistic_tuning_curve, training_loss_curve
from specmine.evaluate import evaluate_model
from specmine.labels import SPEC_ID_COLUMN, read_labels_csv
from specmine.outputs import FigureRun
from specmine.train import (
    ID_COLUMN,
    REPOSITORY_COLUMN,
    prepare_training_data,
    repository_disjoint_split,
    select_model_inputs,
    train_models,
)

SEED = 42
TEST_SIZE = 0.2
BOOTSTRAP_REPLICATES = 2000
LABELS_CSV = project_root / "data" / "labeled_data" / "real_labels.csv"
FIGURE_DIR = project_root / "notebooks" / "figures"
DATA_DIR = project_root / "data"
TEXT_FEATURE = "n_words"
LOCAL_DATA_ONLY = False
MODEL_NAMES = ("logistic_regression", "random_forest", "naive_bayes", "uniform_guess", "markdown_plus_feature")
MODEL_COLORS = {
    "logistic_regression": "#2878B5",
    "random_forest": "#D9782D",
    "naive_bayes": "#568C3B",
    "uniform_guess": "#777777",
    "markdown_plus_feature": "#8E5AA6",
}
MODEL_LABELS = {
    "logistic_regression": "Logistic Regression (features)",
    "random_forest": "Random Forest (features)",
    "naive_bayes": "Naive Bayes (Markdown)",
    "uniform_guess": "Uniform 50/50",
    "markdown_plus_feature": f"Logistic Regression (Markdown + {TEXT_FEATURE})",
}
LABEL_COLORS = {0: "#3B6FB6", 1: "#D9782D"}
GENERATED_FIGURES = []
figure_run = None


def save_figure(fig, name):
    global figure_run
    if figure_run is None:
        figure_run = FigureRun(FIGURE_DIR)
    figure_run.save(fig, name)
    plt.show()


def add_sample_note(fig, frames):
    parts = []
    for name, frame in frames.items():
        counts = frame["label"].value_counts()
        parts.append(
            f"{name}: n={len(frame)}, repos={frame[REPOSITORY_COLUMN].nunique()}, "
            f"label 0={counts.get(0, 0)}, label 1={counts.get(1, 0)}"
        )
    fig.text(0.01, 0.01, " | ".join(parts), ha="left", va="bottom", fontsize=8)
    fig.subplots_adjust(bottom=0.16)


plt.rcParams.update(
    {
        "font.family": "DejaVu Sans",
        "font.size": 10,
        "axes.titlesize": 12,
        "axes.labelsize": 10,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "figure.dpi": 120,
        "savefig.dpi": 300,
    }
)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Labels: {LABELS_CSV.relative_to(project_root)}")
print(f"Figures: {FIGURE_DIR.relative_to(project_root)}")


## 1. Dataset composition

The common analysis sample comprises matched binary labels with readable
Markdown and usable features. Blank labels and excluded records are not included
in the model comparisons. Repository details are available in
`repository_class_counts` rather than printed on every figure.


In [ ]:
def load_metadata():
    return load_data(seed=SEED, data_dir=DATA_DIR, local_data_only=LOCAL_DATA_ONLY)


def load_labeled_features(spec_ids):
    return load_feature_data(
        pd.DataFrame({ID_COLUMN: spec_ids}),
        data_dir=DATA_DIR,
        local_data_only=LOCAL_DATA_ONLY,
    )


labels_frame = read_labels_csv(LABELS_CSV)
metadata = load_metadata()
if metadata is None:
    raise RuntimeError("SpecMine metadata loading failed.")
labeled_ids = labels_frame.loc[labels_frame["label"].notna(), SPEC_ID_COLUMN]
raw_labeled_features = load_labeled_features(labeled_ids)
training_features, training_labels, training_repositories, training_confidences = (
    prepare_training_data(
        metadata,
        labels_frame,
        feature_data=raw_labeled_features,
        seed=SEED,
        include_markdown=True,
        data_dir=DATA_DIR,
        local_data_only=LOCAL_DATA_ONLY,
    )
)
available_ids = set(training_features[ID_COLUMN].astype('string'))
metadata_lookup = metadata[[ID_COLUMN, REPOSITORY_COLUMN]].drop_duplicates()
matched_labels = labels_frame.merge(
    metadata_lookup,
    how="left",
    left_on=SPEC_ID_COLUMN,
    right_on=ID_COLUMN,
    validate="one_to_one",
)
unmatched_label_count = int(matched_labels[REPOSITORY_COLUMN].isna().sum())
if unmatched_label_count:
    print(f"Labels not matched to metadata: {unmatched_label_count}")
matched = matched_labels[REPOSITORY_COLUMN].notna()
included = matched_labels[SPEC_ID_COLUMN].astype('string').isin(available_ids)
excluded = matched_labels.loc[matched & matched_labels["label"].notna() & ~included]
print(f"Excluded labeled specs without readable Markdown: {len(excluded)}")
composition = matched_labels.loc[matched & (included | matched_labels["label"].isna())].copy()
composition["class_name"] = (
    composition["label"].map({0: "Label 0", 1: "Label 1"}).fillna("Ambiguous")
)
repository_class_counts = (
    composition.groupby([REPOSITORY_COLUMN, "class_name"], observed=False)
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["Label 0", "Label 1", "Ambiguous"], fill_value=0)
    .sort_index()
)
print(f"Repository detail is available in repository_class_counts ({len(repository_class_counts)} rows).")
display(
    composition["class_name"]
    .value_counts()
    .reindex(["Label 0", "Label 1", "Ambiguous"], fill_value=0)
    .rename("n")
    .rename_axis("class")
    .to_frame()
)

binary_composition = composition.loc[composition["label"].notna()].copy()
binary_composition["label"] = binary_composition["label"].astype(int)


Class counts use the included common sample. Repository sizes summarize the
distribution without plotting hundreds of individual repository names.


In [ ]:
class_counts = binary_composition["label"].value_counts().reindex([0, 1], fill_value=0)
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(
    ["Label 0", "Label 1"], class_counts.to_numpy(),
    color=[LABEL_COLORS[0], LABEL_COLORS[1]], width=0.55,
)
ax.bar_label(bars, padding=3)
ax.set(ylabel="Number of specs", title="Included labeled specs")
ax.set_ylim(0, max(class_counts.max() * 1.2, 1))
ax.spines[["top", "right"]].set_visible(False)
add_sample_note(fig, {"included": binary_composition})
save_figure(fig, "dataset_label_counts")


Repository sizes are grouped into interpretable bins. Counts and proportions
are per repository, not per specification.


In [ ]:
repo_sizes = binary_composition.groupby(REPOSITORY_COLUMN).size()
repo_size_counts = pd.Series(
    [repo_sizes.eq(1).sum(), repo_sizes.eq(2).sum(),
     repo_sizes.between(3, 5).sum(), repo_sizes.ge(6).sum()],
    index=["1", "2", "3–5", "6+"], name="repositories",
)
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(repo_size_counts.index, repo_size_counts.values, color="#527A9E", width=0.6)
ax.bar_label(bars, padding=3)
ax.set(xlabel="Included specs per repository", ylabel="Number of repositories",
       title="Repository representation")
ax.set_ylim(0, max(repo_size_counts.max() * 1.2, 1))
ax.spines[["top", "right"]].set_visible(False)
add_sample_note(fig, {"included": binary_composition})
save_figure(fig, "repository_size_distribution")


## 2. Repository-disjoint split

The fixed seed and split settings are shared by training and the analysis.
Repository overlap is checked explicitly; counts are shown instead of long
repository lists.


In [ ]:
train_indices, test_indices = repository_disjoint_split(
    training_features,
    training_labels,
    training_repositories,
    test_size=TEST_SIZE,
    seed=SEED,
)
training_results = train_models(
    training_features,
    training_labels,
    training_repositories,
    confidence_scores=training_confidences,
    model_names=MODEL_NAMES,
    text_feature=TEXT_FEATURE,
    test_size=TEST_SIZE,
    seed=SEED,
)
models = {name: result["model"] for name, result in training_results.items()}

train_features = training_features.iloc[train_indices].reset_index(drop=True)
test_features = training_features.iloc[test_indices].reset_index(drop=True)
train_frame = train_features.copy()
train_frame["label"] = training_labels.iloc[train_indices].to_numpy()
train_frame["confidence"] = training_confidences.iloc[train_indices].to_numpy()
train_frame[REPOSITORY_COLUMN] = training_repositories.iloc[train_indices].to_numpy()
test_frame = test_features.copy()
test_frame["label"] = training_labels.iloc[test_indices].to_numpy()
test_frame["confidence"] = training_confidences.iloc[test_indices].to_numpy()
test_frame[REPOSITORY_COLUMN] = training_repositories.iloc[test_indices].to_numpy()

train_repos = sorted(train_frame[REPOSITORY_COLUMN].unique(), key=str)
test_repos = sorted(test_frame[REPOSITORY_COLUMN].unique(), key=str)
shared_repos = set(train_repos) & set(test_repos)
assert not shared_repos, f"Repository leakage detected: {sorted(shared_repos)}"
print(f"Shared repositories: {len(shared_repos)}")
for name, result in training_results.items():
    expected_ids = test_frame[ID_COLUMN].tolist()
    if [row["id"] for row in result["predictions"]] != expected_ids:
        raise RuntimeError(f"{name} did not use the expected holdout rows")
split_balance = pd.DataFrame(
    {
        "train": train_frame["label"].value_counts().reindex([0, 1], fill_value=0),
        "test": test_frame["label"].value_counts().reindex([0, 1], fill_value=0),
    }
).rename_axis("label")
display(split_balance)
print(f"Train: {len(train_frame)} specs / {len(train_repos)} repos")
print(f"Test:  {len(test_frame)} specs / {len(test_repos)} repos")


This grouped bar chart shows how repository holdout changes class counts. It reports counts rather than normalized proportions so the small test support is not obscured.

In [ ]:
positions = np.arange(2)
width = 0.36
fig, ax = plt.subplots(figsize=(7, 4.5))
for offset, split_name, color, frame in [
    (-width / 2, "Train", "#527A9E", train_frame),
    (width / 2, "Test", "#D9782D", test_frame),
]:
    values = frame["label"].value_counts().reindex([0, 1], fill_value=0).to_numpy()
    bars = ax.bar(positions + offset, values, width, label=split_name, color=color)
    ax.bar_label(bars, labels=[f"n={value}" for value in values], padding=2, fontsize=8)
ax.set_xticks(positions, ["Label 0", "Label 1"])
ax.set_ylabel("Number of specs")
ax.set_title("Class balance in the repository-disjoint split")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
add_sample_note(fig, {"train": train_frame, "test": test_frame})
save_figure(fig, "split_class_balance")


## 3. Model comparison

All models use the same test rows. Trainable models use confidence sample weights; the uniform guesser ignores input values and samples each label with probability 0.5, while reporting a positive-class probability of 0.5. Error bars are percentile 95% repository-cluster bootstrap intervals. With few repositories they are descriptive and may be imprecise. AUC is omitted when a resample has only one class.


In [ ]:
evaluation_results = {}
prediction_frames = {}
for model_name, model in models.items():
    evaluated = evaluate_model(model, test_frame)
    if evaluated["metrics"] is None:
        raise RuntimeError(f"Evaluation did not return metrics for {model_name}.")
    evaluation_results[model_name] = evaluated
    prediction_frames[model_name] = pd.DataFrame(evaluated["predictions"])


def metric_values(y_true, y_pred, y_prob):
    values = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": np.nan,
    }
    if np.unique(y_true).size == 2:
        values["roc_auc"] = roc_auc_score(y_true, y_prob)
    return values


def bootstrap_intervals(y_true, y_pred, y_prob, groups, *, replicates, seed):
    rng = np.random.default_rng(seed)
    groups = np.asarray(groups)
    if len(groups) != len(y_true) or len(y_true) == 0:
        raise ValueError("groups and labels must have the same nonzero length")
    group_rows = [np.flatnonzero(groups == group) for group in pd.unique(groups)]
    draws = {name: [] for name in ["accuracy", "precision", "recall", "f1", "roc_auc"]}
    for _ in range(replicates):
        sampled_groups = rng.integers(0, len(group_rows), size=len(group_rows))
        indices = np.concatenate([group_rows[index] for index in sampled_groups])
        sample_metrics = metric_values(
            y_true[indices], y_pred[indices], y_prob[indices]
        )
        for name, value in sample_metrics.items():
            if np.isfinite(value):
                draws[name].append(value)
    return {
        name: tuple(np.quantile(values, [0.025, 0.975])) if values else (np.nan, np.nan)
        for name, values in draws.items()
    }


summary_rows = []
bootstrap_results = {}
metric_names = ["accuracy", "precision", "recall", "f1", "roc_auc"]
for model_index, model_name in enumerate(MODEL_NAMES):
    prediction = prediction_frames[model_name]
    y_true = prediction["y_true"].to_numpy(dtype=int)
    y_pred = prediction["y_pred"].to_numpy(dtype=int)
    y_prob = prediction["y_prob"].to_numpy(dtype=float)
    metrics = evaluation_results[model_name]["metrics"]
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    intervals = bootstrap_intervals(
        y_true,
        y_pred,
        y_prob,
        test_frame[REPOSITORY_COLUMN].to_numpy(),
        replicates=BOOTSTRAP_REPLICATES,
        seed=SEED + model_index,
    )
    bootstrap_results[model_name] = intervals
    row = {
        "model": model_name,
        "n_test": len(prediction),
        "TP": int(tp),
        "FP": int(fp),
        "TN": int(tn),
        "FN": int(fn),
    }
    row.update({name: metrics[name] for name in metric_names})
    row.update({f"{name}_95%_CI": intervals[name] for name in metric_names})
    summary_rows.append(row)
summary_table = pd.DataFrame(summary_rows).set_index("model")
display(summary_table)
print(
    f"Bootstrap replicates: {BOOTSTRAP_REPLICATES}; test labels: {test_frame['label'].value_counts().sort_index().to_dict()}"
)

Points and whiskers show the held-out estimate and 95% repository-cluster
bootstrap percentile interval. The interval describes variation in this
holdout, not uncertainty from refitting or label errors.


In [ ]:
fig, axes = plt.subplots(1, len(metric_names), figsize=(14, 4.7), sharey=True)
positions = np.arange(len(MODEL_NAMES))
for ax, metric in zip(axes, metric_names):
    for position, model_name in enumerate(MODEL_NAMES):
        point = evaluation_results[model_name]["metrics"][metric]
        low, high = bootstrap_results[model_name][metric]
        if point is not None and np.isfinite(point):
            if np.isfinite(low) and np.isfinite(high):
                ax.plot([low, high], [position, position], color=MODEL_COLORS[model_name], lw=2)
            ax.scatter(point, position, color=MODEL_COLORS[model_name], s=35, zorder=3)
    ax.set(title=metric.replace("_", " ").upper(), xlim=(-0.03, 1.03), xlabel="Score")
    ax.grid(axis="x", alpha=0.2)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_yticks(positions, [MODEL_LABELS[name] for name in MODEL_NAMES])
axes[0].invert_yaxis()
fig.suptitle("Model agreement with held-out labels")
fig.tight_layout(rect=(0, 0.11, 1, 0.93))
add_sample_note(fig, {"test": test_frame})
save_figure(fig, "model_metric_comparison")


## 4. Confusion matrices

The heatmaps show raw test-set counts (supplied labels on rows, predictions on columns), preserving the practical scale of each disagreement type.

In [ ]:
ncols = 3
nrows = int(np.ceil(len(MODEL_NAMES) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4.5 * ncols, 4 * nrows), squeeze=False)
for ax, model_name in zip(axes.flat, MODEL_NAMES):
    matrix = np.asarray(evaluation_results[model_name]["metrics"]["confusion_matrix"])
    ax.imshow(matrix, cmap="Blues", vmin=0)
    for row in range(2):
        for column in range(2):
            color = "white" if matrix[row, column] > matrix.max() / 2 else "black"
            ax.text(column, row, str(matrix[row, column]), ha="center", va="center", color=color)
    ax.set_xticks([0, 1], ["Pred 0", "Pred 1"])
    ax.set_yticks([0, 1], ["Label 0", "Label 1"])
    ax.set_title(MODEL_LABELS[model_name], fontsize=9)
for ax in list(axes.flat)[len(MODEL_NAMES):]:
    ax.axis("off")
fig.suptitle("Confusion matrices (raw counts)")
fig.tight_layout(rect=(0, 0.10, 1, 0.95))
add_sample_note(fig, {"test": test_frame})
save_figure(fig, "confusion_matrices")


## 5. ROC and precision–recall curves

The ROC plot uses the diagonal no-skill reference; the precision–recall plot uses test prevalence as its no-skill reference. PR performance is especially informative when the test labels are imbalanced. Curves are skipped with an explicit note if the held-out split contains only one class.

In [ ]:
y_test = test_frame["label"].to_numpy(dtype=int)
if np.unique(y_test).size == 2:
    fig, ax = plt.subplots(figsize=(9, 5.5))
    for model_name in MODEL_NAMES:
        prediction = prediction_frames[model_name]
        fpr, tpr, _ = roc_curve(y_test, prediction["y_prob"].to_numpy())
        auc = evaluation_results[model_name]["metrics"]["roc_auc"]
        ax.plot(
            fpr,
            tpr,
            lw=2,
            color=MODEL_COLORS[model_name],
            label=f"{model_name.replace('_', ' ').title()} (AUC={auc:.2f})",
        )
    ax.plot([0, 1], [0, 1], linestyle="--", color="#666666", label="No-skill")
    ax.set(
        xlabel="False positive rate",
        ylabel="True positive rate",
        title="ROC curves on the held-out test split",
        xlim=(0, 1),
        ylim=(0, 1.02),
    )
    ax.legend(frameon=False, loc="upper left", bbox_to_anchor=(1.02, 1))
    fig.subplots_adjust(right=0.68)
    ax.spines[["top", "right"]].set_visible(False)
    add_sample_note(fig, {"test": test_frame})
    save_figure(fig, "roc_curves")
else:
    print("ROC curves are undefined: the repository-disjoint test split has one class.")


In [ ]:
if np.unique(y_test).size == 2:
    fig, ax = plt.subplots(figsize=(9, 5.5))
    for model_name in MODEL_NAMES:
        prediction = prediction_frames[model_name]
        precision, recall, _ = precision_recall_curve(
            y_test, prediction["y_prob"].to_numpy()
        )
        average_precision = average_precision_score(
            y_test, prediction["y_prob"].to_numpy()
        )
        ax.plot(
            recall,
            precision,
            lw=2,
            color=MODEL_COLORS[model_name],
            label=f"{model_name.replace('_', ' ').title()} (AP={average_precision:.2f})",
        )
    prevalence = float(np.mean(y_test))
    ax.axhline(
        prevalence,
        linestyle="--",
        color="#666666",
        label=f"No-skill (prevalence={prevalence:.2f})",
    )
    ax.set(
        xlabel="Recall",
        ylabel="Precision",
        title="Precision–recall curves on the held-out test split",
        xlim=(0, 1),
        ylim=(0, 1.02),
    )
    ax.legend(frameon=False, loc="upper left", bbox_to_anchor=(1.02, 1))
    fig.subplots_adjust(right=0.68)
    ax.spines[["top", "right"]].set_visible(False)
    add_sample_note(fig, {"test": test_frame})
    save_figure(fig, "precision_recall_curves")
else:
    print(
        "Precision–recall curves are undefined: the repository-disjoint test split has one class."
    )


## 6. Labeler confidence and model scores

Labeler confidence is the labeler's self-reported certainty in its assigned
label (0–100), used as a training weight after division by 100. It is not a
verified probability. Model scores are held-out predicted probabilities of
label 1, not the labeler's confidence; their reliability is not guaranteed.
The uniform baseline returns 0.5 for every score while sampling labels.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
confidence_bins = np.linspace(0, 100, 11)
for label in [0, 1]:
    values = binary_composition.loc[binary_composition["label"].eq(label), "confidence"]
    ax.hist(values, bins=confidence_bins, histtype="step", linewidth=2,
            color=LABEL_COLORS[label], label=f"Label {label} (n={len(values)})")
ax.set(xlabel="Labeler-reported confidence (%)", ylabel="Number of specs",
       title="Confidence in supplied labels", xlim=(0, 100))
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
add_sample_note(fig, {"included": binary_composition})
save_figure(fig, "labeler_confidence_distribution")

fig, axes = plt.subplots(len(MODEL_NAMES), 1, figsize=(7, 9), sharex=True)
for ax, model_name in zip(axes, MODEL_NAMES):
    predictions = prediction_frames[model_name]
    for label in [0, 1]:
        scores = predictions.loc[predictions["y_true"].eq(label), "y_prob"]
        ax.hist(scores, bins=np.linspace(0, 1, 11), histtype="step", linewidth=1.8,
                color=LABEL_COLORS[label], label=f"Label {label}")
    ax.set_ylabel("Count")
    ax.set_title(MODEL_LABELS[model_name], loc="left", fontsize=9)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].legend(frameon=False, loc="upper right", ncol=2)
axes[-1].set_xlabel("Model probability of label 1")
axes[-1].set_xlim(0, 1)
fig.suptitle("Held-out scores by supplied label", y=0.995)
fig.tight_layout(rect=(0, 0.05, 1, 0.98))
add_sample_note(fig, {"test": test_frame})
save_figure(fig, "model_score_distributions")


In [ ]:
test_class_counts = test_frame["label"].value_counts()
if len(test_frame) >= 20 and min(test_class_counts.get(0, 0), test_class_counts.get(1, 0)) >= 4:
    n_bins = min(4, max(2, len(test_frame) // 4))
    fig, ax = plt.subplots(figsize=(9, 5.5))
    ax.plot([0, 1], [0, 1], linestyle="--", color="#666666", label="Ideal agreement")
    for model_name in MODEL_NAMES:
        predictions = prediction_frames[model_name]
        observed, predicted = calibration_curve(
            predictions["y_true"], predictions["y_prob"],
            n_bins=n_bins, strategy="quantile",
        )
        ax.plot(predicted, observed, marker="o", color=MODEL_COLORS[model_name],
                label=MODEL_LABELS[model_name])
    ax.set(xlabel="Mean predicted probability of label 1",
           ylabel="Observed fraction assigned label 1",
           title=f"Held-out score calibration (up to {n_bins} quantile bins)",
           xlim=(0, 1), ylim=(0, 1))
    ax.legend(frameon=False, fontsize=8, loc="upper left", bbox_to_anchor=(1.02, 1))
    fig.subplots_adjust(right=0.68)
    ax.spines[["top", "right"]].set_visible(False)
    add_sample_note(fig, {"test": test_frame})
    save_figure(fig, "model_calibration_curves")
else:
    print("Calibration omitted: require at least 20 test specs and at least 4 from each class.")


## 7. Model signals

These are associations learned on the training split, not causal evidence of
authorship. Logistic coefficients refer to standardized inputs; forest impurity
importance can favor some feature types and correlated features can share
importance. Text models use TF–IDF **character 3–5-grams**, not words: positive
scores favor label 1 and negative scores favor label 0. Learned patterns may
reflect the labeler, repositories, templates, or small-sample variation.


In [ ]:
rf_pipeline = models["random_forest"]
rf_feature_names = rf_pipeline.named_steps["imputer"].get_feature_names_out()
rf_importance = pd.Series(rf_pipeline.named_steps["classifier"].feature_importances_,
                          index=rf_feature_names)
rf_top = rf_importance.nlargest(10).sort_values()
fig, ax = plt.subplots(figsize=(7.5, 5))
ax.barh(rf_top.index, rf_top.values, color=MODEL_COLORS["random_forest"])
ax.set(xlabel="Mean impurity importance", title="Random Forest: strongest numeric features")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout(rect=(0, 0.10, 1, 1))
add_sample_note(fig, {"train": train_frame})
save_figure(fig, "random_forest_top10_importances")


In [ ]:
lr_pipeline = models["logistic_regression"]
lr_feature_names = lr_pipeline.named_steps["imputer"].get_feature_names_out()
lr_coefficients = pd.Series(lr_pipeline.named_steps["classifier"].coef_[0], index=lr_feature_names)
lr_top = lr_coefficients.loc[lr_coefficients.abs().nlargest(10).index].sort_values()
fig, ax = plt.subplots(figsize=(7.5, 5))
ax.barh(lr_top.index, lr_top.values,
        color=[LABEL_COLORS[int(value > 0)] for value in lr_top.values])
ax.axvline(0, color="#333333", linewidth=0.8)
ax.set(xlabel="Coefficient (standardized feature)", title="Logistic Regression: numeric signals")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout(rect=(0, 0.10, 1, 1))
add_sample_note(fig, {"train": train_frame})
save_figure(fig, "logistic_regression_top10_coefficients")


def show_text_signals(model_name):
    pipeline = models[model_name]
    preprocessor = pipeline.named_steps["preprocessor"]
    vectorizer = preprocessor.named_transformers_["markdown"]
    terms = vectorizer.get_feature_names_out()
    classifier = pipeline.named_steps["classifier"]
    if model_name == "naive_bayes":
        weights = classifier.feature_log_prob_[1] - classifier.feature_log_prob_[0]
        axis_label = "Class 1 minus class 0 log probability"
    else:
        weights = classifier.coef_[0][preprocessor.output_indices_["markdown"]]
        axis_label = "Logistic coefficient (TF–IDF character n-gram)"
    if len(terms) != len(weights):
        raise ValueError(f"{model_name} text feature names and weights do not align")
    signal = pd.Series(weights, index=terms)
    selected = pd.concat([signal.nsmallest(8), signal.nlargest(8)]).loc[lambda s: ~s.index.duplicated()].sort_values()
    labels = [repr(term)[1:-1].replace(" ", "␠") for term in selected.index]
    fig, ax = plt.subplots(figsize=(7.5, 6))
    positions = np.arange(len(selected))
    ax.barh(positions, selected.values,
            color=[LABEL_COLORS[int(value > 0)] for value in selected.values])
    ax.set_yticks(positions, labels)
    ax.axvline(0, color="#333333", linewidth=0.8)
    ax.set(xlabel=axis_label, title=MODEL_LABELS[model_name] + ": character signals")
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout(rect=(0, 0.09, 1, 1))
    add_sample_note(fig, {"train": train_frame})
    save_figure(fig, f"{model_name}_character_signals")
    return signal


nb_text_signals = show_text_signals("naive_bayes")
mixed_text_signals = show_text_signals("markdown_plus_feature")
mixed_pipeline = models["markdown_plus_feature"]
mixed_preprocessor = mixed_pipeline.named_steps["preprocessor"]
mixed_numeric_coefficient = float(
    mixed_pipeline.named_steps["classifier"].coef_[0][mixed_preprocessor.output_indices_["numeric"]][0]
)
print(f"Markdown-plus-feature standardized {TEXT_FEATURE} coefficient: {mixed_numeric_coefficient:.3f}")
print("Uniform guess: no learned features; each class has probability 0.5.")


## 8. Error analysis

`prediction_table`, `error_table`, and `repository_class_counts` are available
for detailed inspection without cluttering figures. Confidence is shown as
labeler-reported confidence and model probability separately. Errors mean
disagreement with supplied labels, not confirmed authorship mistakes.


In [ ]:
prediction_rows = []
for model_name in MODEL_NAMES:
    predictions = prediction_frames[model_name].copy()
    predictions["source_repo"] = test_frame[REPOSITORY_COLUMN].to_numpy()
    predictions["labeler_confidence"] = test_frame["confidence"].to_numpy()
    predictions["model"] = model_name
    predictions = predictions.rename(columns={
        "id": "spec_id", "y_true": "true_label", "y_prob": "probability_label_1",
        "y_pred": "predicted_label",
    })
    predictions["predicted_label_confidence"] = np.where(
        predictions["predicted_label"].eq(1),
        predictions["probability_label_1"], 1 - predictions["probability_label_1"],
    )
    prediction_rows.append(predictions[[
        "model", "spec_id", "source_repo", "true_label", "labeler_confidence",
        "predicted_label", "probability_label_1", "predicted_label_confidence",
    ]])
prediction_table = pd.concat(prediction_rows, ignore_index=True)
error_table = prediction_table.loc[
    prediction_table["true_label"].ne(prediction_table["predicted_label"])
].copy()
error_summary = prediction_table.assign(
    disagreement=prediction_table["true_label"].ne(prediction_table["predicted_label"])
).groupby("model")["disagreement"].agg(["sum", "count", "mean"])
display(error_summary.rename(columns={"sum": "disagreements", "count": "n_test", "mean": "rate"}))
print(f"Full prediction and error tables: {len(prediction_table)} and {len(error_table)} rows.")
print("Figures are staged until the final publication cell succeeds.")


## 9. Training diagnostics and exploratory tuning

The baseline estimators do not expose training loss per iteration. The loss
curve below is a separate SGD logistic classifier, not the evaluated baseline;
it shows confidence- and class-weighted training log loss without the penalty.
The C search uses three repository-disjoint, stratified training folds and
unweighted validation F1. It never uses the held-out test set or changes the
baseline models. Fold variation is descriptive, not a confidence interval;
small or imbalanced datasets may not support all folds.


In [ ]:
numeric_columns = training_results["logistic_regression"]["feature_columns"]
diagnostic_features = select_model_inputs(train_features, numeric_columns)
loss_history = training_loss_curve(
    diagnostic_features, train_frame["label"], train_frame["confidence"], seed=SEED
)
fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.plot(loss_history["epoch"], loss_history["training_log_loss"], color=MODEL_COLORS["logistic_regression"])
ax.set(xlabel="Training epoch", ylabel="Weighted training log loss",
       title="SGD logistic diagnostic (training only)")
ax.spines[["top", "right"]].set_visible(False)
add_sample_note(fig, {"train": train_frame})
save_figure(fig, "training_loss_diagnostic")

try:
    tuning_results, tuning_splits = logistic_tuning_curve(
        diagnostic_features, train_frame["label"], train_frame[REPOSITORY_COLUMN],
        train_frame["confidence"], models["logistic_regression"], seed=SEED,
    )
except InsufficientFoldSupport as exc:
    tuning_results, tuning_splits = None, None
    print(f"Tuning plot omitted: {exc}.")
else:
    display(tuning_results)
    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    ax.errorbar(tuning_results["C"], tuning_results["mean_f1"],
                yerr=tuning_results["std_f1"], marker="o", capsize=3,
                color=MODEL_COLORS["logistic_regression"])
    ax.set(xscale="log", xlabel="Logistic regression C", ylabel="Validation F1",
           title="Repository-grouped tuning (mean ± fold SD)", ylim=(-0.05, 1.05))
    ax.spines[["top", "right"]].set_visible(False)
    add_sample_note(fig, {"train": train_frame})
    save_figure(fig, "logistic_tuning_cv")


## 10. Publish this run's figures

Run this final cell only after all analyses have succeeded. Each complete
run is published to its own directory; interrupted staging directories are
not published and can be removed after the notebook has stopped.


In [ ]:
if figure_run is None:
    raise RuntimeError("No figures have been staged; run the analysis cells first.")
GENERATED_FIGURES = figure_run.publish()
print(f"Published {len(GENERATED_FIGURES)} figure files to {figure_run.destination}")
